# V6.5-12 — Resume AdamW trên Studio
Chạy lần lượt CONFIG → CHECK → TRAIN → EVAL → GATE. Resume optimizer/scaler từ checkpoint transfer, giữ tổng 75 epochs.


In [1]:
# CONFIG + restore local artifacts, không tải lại dataset
from pathlib import Path
import json, shutil, sys
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Uav-Powerline-Defect-Detection"]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
DATASET = PROJECT.parent / "datasets/v6p5_roboflow_ablation"
DATA_YAML = DATASET / "data.yaml"
MODEL = PROJECT / "legacy/notebooks/v6p5/yolo11n.pt"
PREV_BEST = PROJECT / "artifacts/v6p5_train_local/20261009T024019Z_roboflow/runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt"
PACK = PROJECT / "data/cache/studio_v6p5_pull/resume_pack/checkpoints"
RUN_ID = "20261009T131324Z_adamw"
RUN_NAME = "yolo11n_v6p5_roboflow_adamw_cosine_640"
ROOT = PROJECT / "artifacts/v6p5_train_local" / RUN_ID
RUN_DIR = ROOT / "runs" / RUN_NAME
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
IMGSZ, EPOCHS, BATCH, DEVICE, WORKERS, SEED = 640, 75, 8, 0, 0, 20261007
for source, dest in [
    (PACK / "adamw_last_epoch29.pt", RUN_DIR / "weights/last.pt"),
    (PACK / "adamw_best_through_epoch29.pt", RUN_DIR / "weights/best.pt"),
    (PACK / "adamw_results_through_epoch29.csv", RUN_DIR / "results.csv"),
]:
    if not dest.exists():
        if not source.is_file(): raise FileNotFoundError(source)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, dest)
for required in [DATA_YAML, MODEL, PREV_BEST, RUN_DIR / "weights/last.pt"]:
    if not required.is_file(): raise FileNotFoundError(required)
print("Resume run:", RUN_ID)
print("Output:", RUN_DIR)


Resume run: 20261009T131324Z_adamw
Output: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/artifacts/v6p5_train_local/20261009T131324Z_adamw/runs/yolo11n_v6p5_roboflow_adamw_cosine_640


In [2]:
# 1. GPU + DATASET GATE (NO TRAINING)
import yaml
import torch
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa khả dụng trong kernel WSL.")
if not 0 <= DEVICE < torch.cuda.device_count():
    raise ValueError(f"DEVICE={DEVICE} không hợp lệ.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(
        f"Cần ultralytics==8.4.172 để so sánh công bằng; hiện tại={ultralytics.__version__}"
    )
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg_names = cfg.get("names", {})
dataset_names = ([cfg_names[k] for k in sorted(cfg_names, key=int)]
                 if isinstance(cfg_names, dict) else list(cfg_names))
if dataset_names != NAMES:
    raise ValueError(f"Taxonomy mismatch: {dataset_names}")
dataset_root = Path(cfg["path"])
for split in ["train", "val"]:
    image_dir = dataset_root / cfg[split]
    if not image_dir.is_dir():
        raise FileNotFoundError(image_dir)
    print(split, len([p for p in image_dir.iterdir() if p.is_file()]))
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)

# Kiểm tra trạng thái checkpoint trước khi train
last = RUN_DIR / "weights/last.pt"
ckpt = torch.load(last, map_location="cpu", weights_only=False)
if ckpt.get("epoch", -1) < 0 or not ckpt.get("optimizer"):
    raise RuntimeError("Checkpoint không còn optimizer/epoch để resume.")
args = ckpt["train_args"]
assert args["optimizer"] == "AdamW" and args["cos_lr"] is True
assert args["epochs"] == EPOCHS and args["batch"] == BATCH
print("Checkpoint epoch index:", ckpt["epoch"])
print("Epoch tiếp theo hiển thị:", ckpt["epoch"] + 2, "/", EPOCHS)
del ckpt


train 18334
val 2272
GPU: NVIDIA L4
PyTorch: 2.14.0+cu130 | Ultralytics: 8.4.172
Checkpoint epoch index: 29
Epoch tiếp theo hiển thị: 31 / 75


In [ ]:
# TRAIN — resume thật sự, không khởi tạo run mới
from ultralytics import YOLO
last = RUN_DIR / "weights/last.pt"
model = YOLO(str(last))
# 8.4.172 cho phép override data và save_dir khi resume.
# save_dir tránh ghi vào đường dẫn /mnt/c của máy cũ.
model.train(resume=True, data=str(DATA_YAML), save_dir=str(RUN_DIR),
            device=DEVICE, workers=WORKERS)
best = RUN_DIR / "weights/best.pt"
if not best.is_file(): raise FileNotFoundError(best)
print("Best:", best)


New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.172 🚀 Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/teamspace/studios/this_studio/datasets/v6p5_roboflow_ablation/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [ ]:
# 3. FAIR EVAL — ADAMW VS MUSGD ON THE SAME FROZEN VAL
import gc
import pandas as pd

try:
    model.cpu()
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

rows = []
for tag, weights in [("adamw_cosine", best), ("musgd_auto", PREV_BEST)]:
    metrics = YOLO(str(weights)).val(
        data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, plots=False, verbose=False,
        project=str(ROOT / "eval"), name=tag, exist_ok=True,
    ).box
    rows.append({"model": tag, "class": "ALL",
                 "AP50": float(metrics.map50), "AP50_95": float(metrics.map)})
    for cid, ap50, ap in zip(metrics.ap_class_index, metrics.ap50, metrics.ap):
        rows.append({"model": tag, "class": NAMES[int(cid)],
                     "AP50": float(ap50), "AP50_95": float(ap)})
result = pd.DataFrame(rows)
result.to_csv(ROOT / "adamw_vs_musgd_eval.csv", index=False)
display(result.pivot_table(index="class", columns="model",
                           values=["AP50", "AP50_95"]).round(4))


In [ ]:
# 4. DECISION GATE
wide = result.pivot(index="class", columns="model", values=["AP50", "AP50_95"])
def delta(metric, cls):
    return float(wide.loc[cls, (metric, "adamw_cosine")]
                 - wide.loc[cls, (metric, "musgd_auto")])

deltas = {
    "global_AP50": delta("AP50", "ALL"),
    "global_AP50_95": delta("AP50_95", "ALL"),
    "foreign_AP50_95": delta("AP50_95", "foreign_object"),
    "strand_AP50_95": delta("AP50_95", "broken_strand"),
}
gate = {
    "global_not_worse_0p5pt": deltas["global_AP50_95"] >= -0.005,
    "global_improves": deltas["global_AP50_95"] > 0.0,
    "foreign_not_worse": deltas["foreign_AP50_95"] >= 0.0,
    "strand_not_worse": deltas["strand_AP50_95"] >= 0.0,
}
decision = {"run_id": RUN_ID, "deltas_vs_musgd": deltas,
            "gate": gate, "pass": all(gate.values())}
(ROOT / "adamw_gate.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))
print("PASS" if decision["pass"] else "FAIL")
